# Lab 07 — Attention, Memory Traffic, and Tiling
**Goal:** Visualize why naive attention materializes a large attention matrix and understand the core idea behind FlashAttention-style tiling.

Free-first: run on CPU or free Colab where possible.

Method: **predict → run → inspect → break → decide**.


In [ ]:
import torch
def attention_memory(B,H,T,D,dtype_bytes=2):
    score_bytes=B*H*T*T*dtype_bytes
    qkv_bytes=3*B*H*T*D*dtype_bytes
    return score_bytes,qkv_bytes
for T in [512,1024,2048,4096]:
    s,q=attention_memory(1,8,T,64)
    print(T,"score GiB",s/1024**3,"QKV GiB",q/1024**3)


## Predict before running
Predict what happens to the attention-score tensor if sequence length doubles.


In [ ]:
import matplotlib.pyplot as plt
Ts=[256,512,1024,2048,4096]
vals=[attention_memory(1,8,T,64)[0]/1024**3 for T in Ts]
plt.plot(Ts,vals,marker="o")
plt.xlabel("sequence length")
plt.ylabel("attention-score memory (GiB, illustrative)")
plt.title("Naive attention score matrix scales quadratically")
plt.show()


## Explain the result
The score tensor has T² scaling. FlashAttention-style approaches reduce high-bandwidth-memory traffic and avoid materializing the full score matrix, using tiling and online normalization. The key idea is systems-level: same mathematical result, different memory behavior.


In [ ]:
# Thought experiment:
# T -> 2T means the score matrix has 4x as many entries.
# A tiled algorithm avoids materializing the full T x T matrix.
for T in [512,1024,2048,4096,8192]:
    print(T, "relative score entries:", (T/512)**2)


## Critical-thinking answers
Do not describe FlashAttention as “just a faster softmax.” It is a carefully engineered attention implementation. The training lesson is to connect algorithmic complexity to the hardware memory hierarchy.

### Sources
https://arxiv.org/abs/2205.14135


## Deepening: sequence length changes memory

This section upgrades the notebook from a runnable demo to a measurable experiment. Predict before executing.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
import torch, time

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

def naive_attention(L, d=64):
    q = torch.randn(1, 4, L, d, device=device, dtype=dtype)
    k = torch.randn(1, 4, L, d, device=device, dtype=dtype)
    v = torch.randn(1, 4, L, d, device=device, dtype=dtype)
    scores = q @ k.transpose(-2, -1)
    p = torch.softmax(scores / (d**0.5), dim=-1)
    return p @ v

for L in [128, 256, 512, 1024]:
    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    _ = naive_attention(L)
    if device == "cuda": torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()/2**30 if device == "cuda" else float("nan")
    print(f"L={L:4d} sec={time.perf_counter()-t0:.4f} peak_GiB={peak:.3f}")


## Break experiment

Double sequence length repeatedly.

Predict score-matrix growth from O(L²).

Then distinguish model weight memory from attention-score memory and activation memory.

### Report

Explain why memory-efficient attention can matter even when the mathematical attention operation is unchanged.

Use templates/EXPERIMENT_CARD.md to record the run.


## 7. Research exercise — use real length distributions to choose attention kernels


In [ ]:
!pip -q install -U datasets transformers
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

tok=AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
ds=load_dataset("HuggingFaceFW/fineweb","sample-10BT",split="train",streaming=True)
lengths=[]
for row in ds:
    t=row.get("text","")
    if t:
        lengths.append(len(tok.encode(t,add_special_tokens=False)))
    if len(lengths)>=500: break

print("p50/p90/p95/p99:",np.percentile(lengths,[50,90,95,99]).astype(int))
for L in [128,512,1024,2048]:
    score_entries=4*L*L
    print(f"L={L:4d} attention-score elements per batch/head: {score_entries:,}")


<details><summary><strong>Expected findings / answer</strong></summary>

The important pattern is quadratic growth in the materialized attention score tensor. A real corpus gives a realistic length distribution rather than an arbitrary benchmark grid. Efficient attention becomes increasingly valuable in the long-context tail because avoiding unnecessary materialization reduces memory traffic and peak memory. The exact speedup is hardware- and implementation-dependent.
</details>
